In [0]:
import pandas as pd
import numpy as np

# don't display print or show large data - slow 
# use .limit() to explore data (faster and safer than using the full table)
# filter early 
# convert to Pandas only on smaller results 
# reuse data frames (refrence already loaded data instead of loading over and over)

# spark = getting the data 
# pandas = working with the data 
hno = spark.table("workspace.unochadatasets.hpc_hno_2025")
fts = spark.table("workspace.unochadatasets.fts_requirements_funding_global")

# HNO baseline
hno_base_spark = spark.sql("""
SELECT
  `Country ISO3` AS iso3,
  `In Need` AS people_in_need
FROM workspace.unochadatasets.hpc_hno_2025
WHERE `Cluster` = 'ALL'
  AND (`Category` IS NULL OR TRIM(`Category`) = '')
""")

hno_base = hno_base_spark.toPandas()
hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])

# FTS baseline - FIXED: Filter to 2025 to match HNO data
# requirements = how much funding is needed
# funding = how much funding is actually provided
fts_base_spark = spark.sql("""
SELECT
  `countryCode` AS iso3,
  SUM(`requirements`) AS requirements,
  SUM(`funding`) AS funding
FROM workspace.unochadatasets.fts_requirements_funding_global
WHERE `year` = 2025
GROUP BY `countryCode`
""")

fts_base = fts_base_spark.toPandas()
fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])

# Merge - based on country 
df = hno_base.merge(fts_base, on="iso3", how="inner")

# Filter - remove rows where some data is missing 
df = df[(df["people_in_need"] > 0) & (df["requirements"] > 0)].copy()

# Already one row per country from the aggregation in SQL query above
df_country = df.copy()

# Compute baseline
df_country["coverage_ratio"] = df_country["funding"] / df_country["requirements"] # how much of the funding was actually provided 
df_country["coverage_ratio"] = df_country["coverage_ratio"].clip(lower=0, upper=1) 

# basically computes how many people need help and how underfunded it is = how many people are left without help (ca)
df_country["gap_score"] = df_country["people_in_need"] * (1 - df_country["coverage_ratio"])

# Rank
ranked = df_country.sort_values("gap_score", ascending=False).reset_index(drop=True)

final = ranked[[
    "iso3",
    "people_in_need",
    "requirements",
    "funding",
    "coverage_ratio",
    "gap_score"
]].copy()

# use sql to add full country name for frontend 

# Convert pandas DataFrame back to Spark and save as a table
final_spark = spark.createDataFrame(final)
final_spark.write.mode("overwrite").saveAsTable("workspace.unochadatasets.gap_rankings_2025")

print("Data saved to workspace.unochadatasets.gap_rankings_2025")

In [0]:
# Export gap rankings for frontend development
# Write to Unity Catalog Volume (serverless-compatible)

# Use the exports volume
volume_path = "/Volumes/workspace/unochadatasets/exports"

# Export as CSV
csv_path = f"{volume_path}/gap_rankings_2025.csv"
final.to_csv(csv_path, index=False)
print(f"✓ CSV exported to: {csv_path}")

# Export as JSON
json_path = f"{volume_path}/gap_rankings_2025.json"
final.to_json(json_path, orient="records", indent=2)
print(f"✓ JSON exported to: {json_path}")

print(f"\nTo download these files:")
print(f"1. Navigate to Catalog Explorer")
print(f"2. Go to workspace > unochadatasets > exports volume")
print(f"3. Right-click the file and select 'Download'")